In [1]:
# %pip install -q pgserver psycopg2-binary sqlalchemy pandas kagglehub

In [2]:
import pixeltable_pgserver as pgserver # аналог pgserver
from sqlalchemy import create_engine, text
import pandas as pd
import os
import kagglehub


# pgdata буде створена, якщо не існує (батьківська директорія має існувати)
db = pgserver.get_server("./temp", cleanup_mode="delete")

uri = db.get_uri(driver="psycopg2")
print("Connection URI:", uri)
print(db.get_postmaster_info().pgdata)
print(db.get_postmaster_info().port)

engine = create_engine(uri, echo=False, future=True)

with engine.connect() as conn:
    version = conn.execute(text('SELECT version()')).scalar_one()
    print(version)

Connection URI: postgresql+psycopg2://postgres:@/postgres?host=/Users/vladyslavratushnyi/Learning/DB_DS/hw4/temp
/Users/vladyslavratushnyi/Learning/DB_DS/hw4/temp
5432
PostgreSQL 18.4 on aarch64-apple-darwin24.6.0, compiled by Apple clang version 17.0.0 (clang-1700.0.13.5), 64-bit


In [3]:
# кастомна функція для виконання sql
def sql_exec(sql_quary:str, engine=engine, is_file=False, printout=False)->list:
    try:
        if is_file:
            with open(sql_quary, 'r') as f:
                sql_quary = f.read()

        ddl_statements = [
            query.strip() + ";" 
            for query in sql_quary.split(';') 
            if query.strip()]

        with engine.begin() as conn:
            for stmt in ddl_statements:
                output = conn.execute(text(stmt))
                if output.returns_rows and printout:
                    for i in output.mappings().fetchall():
                        print(i)


        return f"{len(ddl_statements)} команд виконано"

    except Exception as e:
        print('Операція не виконана:', e)


def sql_reader(query:str, engine=engine):
    try:
        with engine.connect() as con:
            return pd.read_sql_query(query, con)

    except Exception as e:
        print('Операція не виконана:', e)

# Setup і typed-схема Olist

In [4]:
dataset_dir = kagglehub.dataset_download('olistbr/brazilian-ecommerce', output_dir='./data')
print(dataset_dir)

100%|██████████| 42.6M/42.6M [00:01<00:00, 26.5MB/s]

Extracting files...


./data


In [5]:
files = {
    'olist_customers_raw': 'olist_customers_dataset.csv',
    'olist_orders_raw': 'olist_orders_dataset.csv',
    'olist_order_items_raw': 'olist_order_items_dataset.csv',
    'olist_products_raw': 'olist_products_dataset.csv',
    'olist_sellers_raw': 'olist_sellers_dataset.csv',
    'olist_order_reviews_raw': 'olist_order_reviews_dataset.csv',
}

for table_name, file_name in files.items():
    path = os.path.join(dataset_dir, file_name)
    df = pd.read_csv(path)
    df.to_sql(table_name, engine, if_exists='replace', index=False, chunksize=10_000)
    print(table_name, df.shape)


olist_customers_raw (99441, 5)
olist_orders_raw (99441, 8)
olist_order_items_raw (112650, 7)
olist_products_raw (32951, 9)
olist_sellers_raw (3095, 4)
olist_order_reviews_raw (99224, 7)


In [6]:
print(sql_exec('sql_comnds/com1.sql', is_file=True))

19 команд виконано


In [7]:
print(sql_exec('sql_comnds/com2.sql', is_file=True))

6 команд виконано


In [8]:
query = """
SELECT 'customers' AS table_name, COUNT(*) AS n_rows FROM olist_customers
UNION ALL SELECT 'orders', COUNT(*) FROM olist_orders
UNION ALL SELECT 'order_items', COUNT(*) FROM olist_order_items
UNION ALL SELECT 'products', COUNT(*) FROM olist_products
UNION ALL SELECT 'sellers', COUNT(*) FROM olist_sellers
UNION ALL SELECT 'reviews', COUNT(*) FROM olist_order_reviews
ORDER BY table_name;
"""
sql_reader(query)

,table_name,n_rows
0,customers,99441
1,order_items,112650
2,orders,99441
3,products,32951
4,reviews,99224
5,sellers,3095


# DML-запити з PostgreSQL-фічами

In [9]:
# INSERT ... RETURNING
sql_exec('sql_comnds/com3.sql', is_file=True, printout=True)

{'alert_id': 1, 'seller_id': '4a3ca9315b744ce9f8e9374361493884', 'alert_type': 'late_delivery', 'severity': 4, 'created_at': datetime.datetime(2026, 9, 27, 23, 55, 1, 920343, tzinfo=datetime.timezone(datetime.timedelta(seconds=3600)))}


'3 команд виконано'

Була створена таблиця seller_alerts яка зберігає id продавця з найбільшою кількістю запізнень. 

Внесені до таблиці дані повернені за допомогою RETURNING (нічого не повертає якщо нові рядки не були внесені). Команда є багаторазовою у використанні так яя таблиця seller_alerts видаляється при кожному новому запуску.

In [10]:
# INSERT ... ON CONFLICT DO UPDATE: idempotent upsert агрегованих seller-score
sql_exec('sql_comnds/com4.sql', is_file=True, printout=False)

'3 команд виконано'

**Великий output**

Кількість виведених рядків = 3095

printout=True щоб вивести результат

Обчислює для кожного продавця середній рейтинг (avg_rating) на основі відгуків на його замовлення та кількість врахованих відгуків (n_reviews), визначає tier (gold/silver/bronze) і записує результат у seller_score. При повторному запуску існуючі рядки оновлюються, а не дублюються.

**Чому seller_orders використовує SELECT DISTINCT**

Один seller_id може мати кілька рядків в olist_order_items для одного й того ж order_id. Без DISTINCT пара (seller_id, order_id) дублювалась би стільки разів, скільки товарів продавця в замовленні. DISTINCT гарантує, що кожна пара seller–order враховується рівно один раз.

**Чому review потрібно спочатку агрегувати на рівні order_id**

В olist_order_reviews на одне замовлення теоретично може бути більше одного відгуку. review_per_order зводить це до одного значення на order_id (AVG(review_score) — на випадок кількох оцінок для одного замовлення), і лише після цього результат безпечно приєднується до продавців через seller_orders.

**Чому ON CONFLICT (seller_id) DO UPDATE робить запуск повторюваним**

seller_id — PRIMARY KEY таблиці seller_score. При повторному запуску INSERT для вже існуючого seller_id викликав би порушення унікальності (duplicate key violates unique constraint) і запит впав би з помилкою, або (без ON CONFLICT) довелось би спочатку робити DELETE/TRUNCATE. ON CONFLICT (seller_id) DO UPDATE перетворює це на upsert

**Чому в деяких продавців avg_rating = NULL, n_reviews = 0, tier = NULL**

Це наслідок LEFT JOIN у CTE seller_rating: seller_orders (ліва таблиця) містить усіх продавців, які колись мали хоч один order_item, незалежно від того, чи залишив покупець відгук.

In [11]:
# UPDATE ... FROM: оновлення прапорця запізнення в olist_orders
sql_exec('sql_comnds/com5.sql', is_file=True)

'2 команд виконано'

**Великий output**

Кількість виведених рядків = рядкам з olist_orders

1. ALTER TABLE ... ADD COLUMN IF NOT EXISTS is_late
Додає до olist_orders нову колонку is_late BOOLEAN з дефолтом FALSE. IF NOT EXISTS робить цю частину безпечною для повторного запуску — якщо колонка вже є, помилки не буде.

2. CTE delivery_flags
Для кожного замовлення обчислює, чи було воно доставлене з запізненням: порівнює order_delivered_customer_date з order_estimated_delivery_date (тільки дата, без часу — ::DATE). Якщо доставки ще не було, або немає оціночної дати — is_late_calc = FALSE (через AND, а не через порівняння з NULL, тому логіка коректно повертає FALSE, а не NULL).

3. UPDATE ... FROM
Оновлює is_late в olist_orders, приєднуючи (через FROM ... WHERE) обчислені значення з delivery_flags по order_id. Це стандартний спосіб в PostgreSQL зробити UPDATE на основі результату складнішого обчислення/CTE, а не просто константи.

Ідемпотентність: запит можна запускати скільки завгодно разів — щоразу is_late_calc перераховується заново з тих самих вихідних дат і просто перезаписує те саме значення (is_late буде або лишатись TRUE, або FALSE), без накопичення дублікатів чи побічних ефектів.

RETURNING повертає order_id, customer_id та новий is_late для кожного оновленого рядка — зручно для одразу перевірки, скільки і яких замовлень позначено як прострочені.

In [12]:
# DELETE ... RETURNING: контрольоване видалення застарілих alert-ів
query = """
DELETE FROM seller_alerts
WHERE created_at < NOW() - INTERVAL '30 days'
RETURNING alert_id, seller_id, alert_type, created_at;
"""
sql_reader(query)

,alert_id,seller_id,alert_type,created_at


**Що робить запит:** видаляє з seller_alerts усі рядки, старіші за 30 днів, і через RETURNING повертає дані видалених рядків.

Запит нічого не повернув по причині актуальності радків.

**Ось запит для перевірки:**

In [13]:
query = """
SELECT COUNT(*) FROM seller_alerts;
SELECT alert_id, created_at, NOW() - created_at AS age FROM seller_alerts;
"""
sql_reader(query)

,alert_id,created_at,age
0,1,2026-09-27 22:55:01.920343+00:00,0 days 00:00:01.261137


# CREATE TABLE customer_segments

In [14]:
sql_exec('sql_comnds/com6.sql', is_file=True)

'7 команд виконано'

In [15]:
# Перевірте constraints:
query = """
SELECT
    conname,
    contype,
    pg_get_constraintdef(oid) AS definition
FROM pg_constraint
WHERE conrelid = 'customer_segments'::regclass
ORDER BY conname;
"""
sql_reader(query)

,conname,contype,definition
0,customer_segments_assigned_at_not_null,n,NOT NULL assigned_at
1,customer_segments_customer_unique_id_key,u,UNIQUE (customer_unique_id)
2,customer_segments_customer_unique_id_not_null,n,NOT NULL customer_unique_id
3,customer_segments_monetary_value_check,c,CHECK ((monetary_value >= (0)::numeric))
4,customer_segments_monetary_value_not_null,n,NOT NULL monetary_value
5,customer_segments_n_orders_check,c,CHECK ((n_orders >= 0))
6,customer_segments_n_orders_not_null,n,NOT NULL n_orders
7,customer_segments_pkey,p,PRIMARY KEY (segment_id)
8,customer_segments_recency_days_check,c,CHECK (((recency_days IS NULL) OR (recency_day...
9,customer_segments_rfm_score_check,c,CHECK (((rfm_score >= 1) AND (rfm_score <= 5)))


**Чому customer_segments використовує customer_unique_id, а не customer_id**

Якщо рахувати сегментацію по customer_id, той самий клієнт з 5 замовленнями під 5 різними customer_id буде розбитий на 5 окремих "нових" клієнтів з одним замовленням кожен — це спотворює і n_orders, і monetary_value, і, як наслідок, сам сегмент. customer_unique_id дозволяє коректно агрегувати всю історію покупок однієї людини.

**Чому olist_customer_dim потрібна для FK**

customer_segments посилається через FOREIGN KEY саме на olist_customer_dim(customer_unique_id), а не напряму на olist_customers, тому що в olist_customers немає PRIMARY KEY/UNIQUE на customer_unique_id — там цей стовпець повторюється (по одному рядку на кожен customer_id), тож FK на нього фізично неможливий у PostgreSQL. olist_customer_dim — це дедуплікована вимірна таблиця (одне значення customer_unique_id — один рядок, обраний через DISTINCT ON за найновішим замовленням), яка і надає потрібний унікальний ключ для референсу.

**Чому reference_date краще брати з датасету, а не з CURRENT_DATE**

Якщо рахувати recency_days відносно CURRENT_DATE, то всі клієнти без винятку матимуть recency в тисячі днів і потраплятимуть у inactive/низький RFM-score незалежно від реальної поведінки — сегментація втратить будь-який сенс. Використання MAX(order_purchase_timestamp) + 1 day як "умовного сьогодні" імітує момент, коли аналіз реально проводився б в контексті самого датасету, і зберігає коректну відносну шкалу recency між клієнтами.

**Як ON CONFLICT робить сегментацію повторюваною**

customer_unique_id має UNIQUE-обмеження в customer_segments, тож при повторному запуску скрипта INSERT для вже існуючого клієнта конфліктував би з цим обмеженням. ON CONFLICT (customer_unique_id) DO UPDATE перетворює це на upsert: замість помилки — оновлюються segment_name, rfm_score, recency_days, monetary_value, n_orders та assigned_at = NOW() для існуючого рядка. Це робить запит ідемпотентним щодо структури таблиці, хоча самі значення сегментів, звісно, можуть змінюватись від запуску до запуску, якщо змінились вихідні дані — це очікувано й правильно, адже сегментація має відображати актуальний стан.

# JOIN-запити з business-interpretation

### INNER JOIN: топ категорій за revenue і seller-state

Business-question: які категорії товарів і штати продавців дають найбільший revenue у delivered-замовленнях?

In [16]:
# INNER JOIN: топ категорій за revenue і seller-state
query = """
SELECT
    p.product_category_name,
    s.seller_state,
    COUNT(DISTINCT oi.order_id) AS n_orders,
    SUM(oi.price + oi.freight_value)::NUMERIC(12, 2) AS revenue
FROM olist_order_items AS oi
JOIN olist_orders AS o
    ON o.order_id = oi.order_id
JOIN olist_products AS p
    ON p.product_id = oi.product_id
JOIN olist_sellers AS s
    ON s.seller_id = oi.seller_id
WHERE o.order_status = 'delivered'
GROUP BY p.product_category_name, s.seller_state
ORDER BY revenue DESC NULLS LAST
LIMIT 10;
"""
sql_reader(query)

,product_category_name,seller_state,n_orders,revenue
0,cama_mesa_banho,SP,8214,1070048.14
1,relogios_presentes,SP,4494,1016168.80
2,beleza_saude,SP,5684,790967.66
3,esporte_lazer,SP,4772,680165.54
4,moveis_decoracao,SP,4597,605650.63
5,cool_stuff,SP,2621,484196.53
6,utilidades_domesticas,SP,4021,471810.66
7,automotivo,SP,3018,438833.22
8,informatica_acessorios,SP,2783,388506.42
9,bebes,SP,1981,339129.74


Interpretation: Усі топ-10 позицій за revenue належать продавцям зі штату SP — жоден інший штат навіть не потрапляє в топ-10, що свідчить про сильну географічну концентрацію продавців-лідерів саме в SP.

Лідирує категорія cama_mesa_banho з revenue R$ 1 070 048 та 8214 замовленнями — і за кількістю замовлень, і за сумою це найбільша категорія

### LEFT JOIN + COALESCE: customer-level features без множення review

Business-question: які customer-level features можна побудувати для реального покупця на рівні customer_unique_id?

In [17]:
query = """
WITH order_totals AS (
    SELECT
        o.order_id,
        c.customer_unique_id,
        SUM(oi.price + oi.freight_value)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    JOIN olist_customers AS c
        ON c.customer_id = o.customer_id
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.order_id,
        c.customer_unique_id
),
review_per_order AS (
    SELECT
        rv.order_id,
        AVG(rv.review_score)::NUMERIC(3, 2) AS order_review_score
    FROM olist_order_reviews AS rv
    WHERE rv.review_score IS NOT NULL
    GROUP BY rv.order_id
),
customer_stats AS (
    SELECT
        ot.customer_unique_id,
        COUNT(DISTINCT ot.order_id) AS n_orders,
        COALESCE(SUM(ot.order_total), 0)::NUMERIC(12, 2) AS total_spend,
        AVG(rpo.order_review_score)::NUMERIC(3, 2) AS avg_review_score
    FROM order_totals AS ot
    LEFT JOIN review_per_order AS rpo
        ON rpo.order_id = ot.order_id
    GROUP BY ot.customer_unique_id
)
SELECT
    cd.customer_unique_id,
    cd.customer_state,
    COALESCE(cs.n_orders, 0) AS n_orders,
    COALESCE(cs.total_spend, 0)::NUMERIC(12, 2) AS total_spend,
    COALESCE(cs.avg_review_score, 0)::NUMERIC(3, 2) AS avg_review_score
FROM olist_customer_dim AS cd
LEFT JOIN customer_stats AS cs
    ON cs.customer_unique_id = cd.customer_unique_id
ORDER BY total_spend DESC NULLS LAST
LIMIT 20;

"""
sql_reader(query)

,customer_unique_id,customer_state,n_orders,total_spend,avg_review_score
0,0a0a92112bd4c708ca5fde585afaa872,RJ,1,13664.08,1.0
1,da122df9eeddfedc1dc1f5349a1a690c,RJ,2,7571.63,5.0
2,763c8b1c9c68a0229c42c9fc6f662b93,ES,1,7274.88,1.0
3,dc4802a71eae9be1dd28f5d788ceb526,MS,1,6929.31,5.0
4,459bef486812aa25204be022145caa62,ES,1,6922.21,0.0
5,ff4159b92c40ebe40454e3e6a7c35ed6,SP,1,6726.66,5.0
6,4007669dec559734d6f53e029e360987,MG,1,6081.54,1.0
7,5d0a2980b292d049061542014e8960bf,GO,1,4809.44,1.0
8,eebb5dda148d3893cdaf5b5ca3040ccb,SP,1,4764.34,4.0
9,48e1ac109decbb87765a3eade6854098,PB,1,4681.78,5.0


**Interpretation:** Топ-20 клієнтів за total_spend майже всі мають n_orders = 1 — це разові дорогі покупки, а не лояльні постійники (виняток: RS-клієнт з 4 замовленнями на 4655.88, RJ з 2 на 7571.63). Географія розкидана по штатах (RJ, ES, SP, MG...), без явної концентрації. Помітно, що високий чек не корелює з високим review_score — є клієнти з total_spend >6000 і оцінкою 0–1 (рядки 0, 2, 4, 6), тобто дорога покупка не гарантує задоволеність.

Попереднє агрегування review на рівні order_id (через review_per_order) важливе тут так само, як і в seller_score: без нього avg_review_score рахувався б по рядках order_items, і замовлення з кількома товарами (кількома order_item_id) множило б один і той самий review на кількість позицій, штучно завищуючи його вагу в середньому по клієнту.

### FULL OUTER JOIN: які штати присутні серед клієнтів і продавців

Business-question: які штати представлені тільки серед клієнтів, тільки серед продавців або в обох групах?

In [18]:
query = """
WITH customer_states AS (
    SELECT customer_state AS state, COUNT(*) AS n_customers
    FROM olist_customers
    GROUP BY customer_state
), seller_states AS (
    SELECT seller_state AS state, COUNT(*) AS n_sellers
    FROM olist_sellers
    GROUP BY seller_state
)
SELECT
    COALESCE(c.state, s.state) AS state,
    c.n_customers,
    s.n_sellers,
    CASE
        WHEN c.state IS NOT NULL AND s.state IS NOT NULL THEN 'both'
        WHEN c.state IS NOT NULL THEN 'customers_only'
        ELSE 'sellers_only'
    END AS state_presence
FROM customer_states AS c
FULL OUTER JOIN seller_states AS s
    ON s.state = c.state
ORDER BY state;
"""
sql_reader(query)

,state,n_customers,n_sellers,state_presence
0,AC,81,1.0,both
1,AL,413,NaN,customers_only
2,AM,148,1.0,both
3,AP,68,NaN,customers_only
4,BA,3380,19.0,both
5,CE,1336,13.0,both
6,DF,2140,30.0,both
7,ES,2033,23.0,both
8,GO,2020,40.0,both
9,MA,747,1.0,both


**Interpretation:** З 27 штатів 23 присутні в обох групах, і лише 4 (AL, AP, RR, TO) — тільки серед клієнтів, продавців у них немає взагалі. sellers_only штатів немає жодного — тобто продавці не працюють у регіонах без клієнтів (що логічно). Найсильніший дисбаланс — у SP: 41746 клієнтів проти 1849 продавців (~22:1), що підтверджує концентрацію продавців у SP з попереднього запиту — навіть при величезній кількості своїх покупців SP ще й обслуговує попит з інших штатів.

### SELF JOIN: пари продавців з одного штату, але з різних міст

Business-question: які seller-пари можуть бути кандидатами для регіонального порівняння в межах одного штату?

In [19]:
query = """
SELECT
    s1.seller_id AS seller_a,
    s2.seller_id AS seller_b,
    s1.seller_state,
    s1.seller_city AS city_a,
    s2.seller_city AS city_b
FROM olist_sellers AS s1
JOIN olist_sellers AS s2
    ON s1.seller_state = s2.seller_state
   AND s1.seller_id < s2.seller_id
   AND s1.seller_city IS DISTINCT FROM s2.seller_city
ORDER BY s1.seller_state, seller_a, seller_b
LIMIT 20;
"""
sql_reader(query)

,seller_a,seller_b,seller_state,city_a,city_b
0,1444c08e64d55fb3c25f0f09c07ffcf2,2b402d5dc42554061f8ea98d1916f148,BA,lauro de freitas,irece
1,1444c08e64d55fb3c25f0f09c07ffcf2,398cb257329ef7af7f1943a8974a3cbc,BA,lauro de freitas,salvador
2,1444c08e64d55fb3c25f0f09c07ffcf2,4221a7df464f1fe2955934e30ff3a5a1,BA,lauro de freitas,bahia
3,1444c08e64d55fb3c25f0f09c07ffcf2,43753b27d77860f1654aa72e251a7878,BA,lauro de freitas,barro alto
4,1444c08e64d55fb3c25f0f09c07ffcf2,4aba391bc3b88717ce08eb11e44937b2,BA,lauro de freitas,arraial d'ajuda (porto seguro)
5,1444c08e64d55fb3c25f0f09c07ffcf2,4fb41dff7c50136976d1a5cf004a42e2,BA,lauro de freitas,feira de santana
6,1444c08e64d55fb3c25f0f09c07ffcf2,651530bf5c607240ccdd89a30c9c9712,BA,lauro de freitas,ipira
7,1444c08e64d55fb3c25f0f09c07ffcf2,659e8466eb3ff1b0e8740d74fb7bbedd,BA,lauro de freitas,eunapolis
8,1444c08e64d55fb3c25f0f09c07ffcf2,75d34ebb1bd0bd7dde40dd507b8169c3,BA,lauro de freitas,salvador
9,1444c08e64d55fb3c25f0f09c07ffcf2,a3dd39f583bc80bd8c5901c95878921e,BA,lauro de freitas,salvador


**Interpretation:** Вивід відсортований по seller_state = BA і показує лише пари з продавцем lauro de freitas та irece як seller_a — це початок повного списку комбінацій, а не репрезентативна вибірка (LIMIT або сортування обрізає результат саме на BA). Помітно, що salvador з'являється як city_b найчастіше (рядки 2, 8-10, 14-16) — очікувано, оскільки це найбільше місто BA з найбільшою концентрацією продавців, тож він частіше потрапляє в пари як "інше місто". 

Такі пари корисні для порівняння регіональної конкуренції: продавці з малих міст проти продавців зі столиці штату в межах одного ринку доставки.

### Anti-join: замовлення без відгуку

Business-question: які замовлення не мають жодного review-запису?

In [20]:
query = """
SELECT
    o.order_id,
    o.customer_id,
    o.order_status,
    o.order_purchase_timestamp
FROM olist_orders AS o
LEFT JOIN olist_order_reviews AS rv
    ON rv.order_id = o.order_id
WHERE rv.review_row_id IS NULL
ORDER BY o.order_purchase_timestamp
LIMIT 20;
"""
sql_reader(query)

,order_id,customer_id,order_status,order_purchase_timestamp
0,cda873529ca7ab71f677d5ec11a40304,76c74aaff2f3f7355f46d9818ad092b8,shipped,2016-10-05 16:57:30
1,5da2004d85f2828349d887c989da2adc,95f592086150f0c95d3082aea047b816,delivered,2016-10-05 20:20:59
2,02190241f7190a1f3c7e0df95a749c6a,55c9044401ffab0c717f24405e2e042f,delivered,2016-10-07 10:52:22
3,0efd0bc268d34da3f01f4ff25a6d4335,95446917717bb58d553d107d0f1668f6,shipped,2016-10-07 15:53:31
4,c776863a93dc0740c6e7d78104b21413,1710b798ebdc5db6665c815b90a64999,delivered,2016-10-10 15:01:27
5,d730d8d0ea3149120703933fdf98b2d4,ce2a66464da8b80dedd7639a6e489a58,shipped,2017-01-13 20:12:41
6,05cd156f0f47579a36d7a52724fc31b8,976224a95faf328b17654b1b1411d5e1,delivered,2017-01-14 18:27:28
7,719b940ab2f1e351ffb14a0a02ffe3c4,ede9a1daf58d256c0d648ef7e1479010,delivered,2017-01-16 13:01:51
8,fbd82b1fe98967de0db29b6cf134d00a,45f5c9d984dbfa2105e3544cc85133bf,delivered,2017-01-18 08:55:36
9,171398a0cd843c65ac78aee9e05a1f49,f0244902c07b7c0799e468218579a925,delivered,2017-01-18 16:07:37


**Interpretation:** Переважна більшість замовлень без review мають статус delivered — тобто відсутність review не пов'язана з тим, що замовлення не дійшло, а радше з тим, що покупець просто не залишив відгук. Є й декілька винятків зі статусами shipped та unavailable, де відсутність review логічна — товар ще в дорозі або не доставлений взагалі, тож просити відгук ще/вже нема сенсу.

# GROUP BY, HAVING і агрегати

### Запит: продавці з великою кількістю категорій товарів (STRING_AGG, HAVING)

Business-question: які продавці торгують у 5+ різних категоріях товарів, і які саме це категорії?

In [21]:
query = """
SELECT
    oi.seller_id,
    s.seller_state,
    COUNT(DISTINCT p.product_category_name) AS n_categories,
    STRING_AGG(DISTINCT p.product_category_name, ', ' ORDER BY p.product_category_name) AS categories
FROM olist_order_items AS oi
JOIN olist_products AS p
    ON p.product_id = oi.product_id
JOIN olist_sellers AS s
    ON s.seller_id = oi.seller_id
WHERE p.product_category_name IS NOT NULL
GROUP BY oi.seller_id, s.seller_state
HAVING COUNT(DISTINCT p.product_category_name) >= 5
ORDER BY n_categories DESC
LIMIT 10;
"""
sql_reader(query)

,seller_id,seller_state,n_categories,categories
0,b2ba3715d723d245138f291a6fe42594,SP,27,"agro_industria_e_comercio, audio, automotivo, ..."
1,955fee9216a65b617aa5c0531780ce60,SP,23,"agro_industria_e_comercio, alimentos, artes, a..."
2,4e922959ae960d389249c378d1c939f5,SP,23,"agro_industria_e_comercio, alimentos, artigos_..."
3,1da3aeb70d7989d1e6d9b0e887f97c23,SP,21,"automotivo, bebes, beleza_saude, brinquedos, c..."
4,f8db351d8c4c4c22c6835c19a46f01b0,SP,19,"audio, automotivo, beleza_saude, brinquedos, c..."
5,18a349e75d307f4b4cc646a691ed4216,SP,17,"audio, automotivo, casa_construcao, cine_foto,..."
6,6edacfd9f9074789dad6d62ba7950b9c,SP,15,"automotivo, bebes, brinquedos, cama_mesa_banho..."
7,70a12e78e608ac31179aea7f8422044b,SP,15,"audio, automotivo, bebes, beleza_saude, consol..."
8,8b28d096634035667e8263d57ba3368c,SP,14,"automotivo, bebes, beleza_saude, consoles_game..."
9,2ff97219cb8622eaf3cd89b7d9c09824,SP,14,"artigos_de_natal, bebes, beleza_saude, brinque..."


### Запит: штати-продавці з високим revenue та низькою середньою оцінкою (HAVING)

Business-question: які штати продавців генерують значний revenue (>50000), але при цьому мають середній review_score нижче 4.0 — потенційна зона ризику для якості обслуговування?

In [22]:
query = """
SELECT
    s.seller_state,
    COUNT(DISTINCT oi.order_id) AS n_orders,
    SUM(oi.price + oi.freight_value)::NUMERIC(12,2) AS revenue,
    ROUND(AVG(rv.review_score), 2) AS avg_review_score
FROM olist_order_items AS oi
JOIN olist_sellers AS s
    ON s.seller_id = oi.seller_id
JOIN olist_orders AS o
    ON o.order_id = oi.order_id
LEFT JOIN olist_order_reviews AS rv
    ON rv.order_id = oi.order_id
WHERE o.order_status = 'delivered'
GROUP BY s.seller_state
HAVING SUM(oi.price + oi.freight_value) > 50000
   AND AVG(rv.review_score) < 4.0
ORDER BY revenue DESC;
"""
sql_reader(query)

,seller_state,n_orders,revenue,avg_review_score


**interpretation:** жоден штат з revenue >50000 не має середньої оцінки нижче 4.0 — це свідчить про стабільно високий рівень задоволеності навіть у великих/завантажених регіонах продажів

**WHERE і HAVING**

WHERE працює до GROUP BY/агрегації — він фільтрує окремі рядки з olist_order_items/olist_orders ще на етапі JOIN, до того як вони об'єднаються в групи по seller_state. Тут він відкидає всі рядки, де order_status != 'delivered', — тобто в агрегацію (SUM, AVG) взагалі не потрапляють дані по незавершених/скасованих замовленнях. WHERE не може посилатись на результат агрегатних функцій (SUM(...), AVG(...)), бо на момент його виконання агрегація ще не відбулась.

HAVING працює після GROUP BY — він фільтрує вже готові групи (по одному рядку на seller_state), тобто результат агрегатних функцій. Тут він залишає тільки ті штати, для яких сумарний revenue вже підрахований і перевищує 50000, а середній review_score вже підрахований і менший за 4.0. HAVING не може використовуватись для фільтрації окремих рядків до групування — на цьому етапі окремих рядків з oi/o вже не існує, є тільки згруповані підсумки.

WHERE вирішує, які рядки беруть участь в обчисленні; HAVING вирішує, які вже обчислені групи потраплять у фінальний результат.

### Запит 3 — клієнти з кількома замовленнями та списком їхніх order_id (ARRAY_AGG)

Business-question: які клієнти (customer_unique_id) оформили більше одного замовлення, і які саме order_id їм належать (для подальшого ручного аудиту)?

In [23]:
query = """
SELECT
    c.customer_unique_id,
    COUNT(o.order_id) AS n_orders,
    ARRAY_AGG(o.order_id ORDER BY o.order_purchase_timestamp) AS order_ids,
    SUM(oi.price + oi.freight_value)::NUMERIC(12,2) AS total_spend
FROM olist_customers AS c
JOIN olist_orders AS o
    ON o.customer_id = c.customer_id
JOIN olist_order_items AS oi
    ON oi.order_id = o.order_id
GROUP BY c.customer_unique_id
HAVING COUNT(o.order_id) > 1
ORDER BY n_orders DESC
LIMIT 10;
"""
sql_reader(query)

,customer_unique_id,n_orders,order_ids,total_spend
0,c8460e4251689ba205045f3ea17884a1,24,"[cb1f3a44e8b8527e16913306a4d3de2f, cb1f3a44e8b...",4655.88
1,4546caea018ad8c692964e3382debd19,21,"[8272b63d03f5f79c56e9e4120aec44ef, 8272b63d03f...",196.17
2,c402f431464c72e27330a67f7b94d4fb,20,"[1b15974a0141d54e36626dca3fdc731a, 1b15974a014...",2202.40
3,698e1cf81d01a3d389d96145f7fa6df8,20,"[ab14fdcfbe524636d65ee38360e22ce8, ab14fdcfbe5...",2262.80
4,0f5ac8d5c31de21d2f25e24be15bbffb,18,"[428a2f660dc84138d969ccd69a0ab6d5, 428a2f660dc...",1544.82
5,8d50f5eadf50201ccdcedfb9e2ac8455,16,"[5d848f3d93a493c1c8955e018240e7ca, 369634708db...",902.04
6,11f97da02237a49c8e783dfda6f50e8e,15,"[9ef13efd6949e4573a18964dd1bbe7f5, 9ef13efd694...",783.00
7,eae0a83d752b1dd32697e0e7b4221656,15,"[71dab1155600756af6de79de92e712e3, 71dab115560...",2783.01
8,31e412b9fb766b6794724ed17a41dfa6,14,"[73c8ab38f07dc94389065f7eba4f297a, 73c8ab38f07...",1014.02
9,3e43e6105506432c953e165fb2acf44c,14,"[ed0b6f8c8617370e53735a5d963bb5b6, ed0b6f8c861...",1172.67


# Set operations, підзапит/window і Reflection

In [24]:
query = """
SELECT customer_state AS state, 'customer' AS role
FROM olist_customers

UNION

SELECT seller_state AS state, 'seller' AS role
FROM olist_sellers
ORDER BY state, role;
"""
sql_reader(query)

,state,role
0,AC,customer
1,AC,seller
2,AL,customer
3,AM,customer
4,AM,seller
5,AP,customer
6,BA,customer
7,BA,seller
8,CE,customer
9,CE,seller


UNION — тут кожен вихідний рядок вже унікальний за змістом сам по собі: пара (state, role) має цікавити нас лише як факт присутності, а не як лічильник. UNION ALL зберіг би мільйони дублікатів, що тут не потрібно — нас цікавить множина унікальних комбінацій штат+роль, а не підрахунок.

INTERSECT показав би тільки те, що спільне між двома SELECT, але тут результати мають різну структуру за змістом, тож INTERSECT між ними технічно не дасть змістовного перетину ролей — він працював би, лише якщо порівнювати список самих state без колонки role.

EXCEPT показав би різницю множин (наприклад, "штати клієнтів мінус штати продавців"), у цьому запиті потрібне саме об'єднання, а не віднімання.

Отже UNION тут обраний тому, що завдання — об'єднати дві множини записів у одну зведену таблицю без дублікатів, зберігаючи інформацію про роль (customer/seller) як частину самого рядка.

# Reflection

Найскладнішим типом join виявився FULL OUTER JOIN — на відміну від INNER/LEFT JOIN, тут треба одночасно тримати в голові три сценарії (є в обох таблицях, тільки зліва, тільки справа) і правильно обробляти NULL з обох сторін через COALESCE, інакше легко втратити частину даних або неправильно інтерпретувати результат.

Найскладнішим було сформулювати business-question взагалі для всіх запитів — бракує досвіду переходити від "які дані є" до "яке реальне бізнес-питання ці дані відповідають", особливо для set-operations та self-join.

З PostgreSQL-фіч найкориснішою для production-ETL здається ON CONFLICT — саме вона робить пайплайни безпечними для повторного запуску без ручного DELETE/TRUNCATE.

Window functions я б використав для recency/frequency-фіч у реальному ML-проєкті: ROW_NUMBER() OVER (PARTITION BY customer_unique_id ORDER BY order_purchase_timestamp) для номера замовлення, LAG() для інтервалу між покупками, SUM() OVER для running total spend.

Data-quality проблеми: по перше це згадана в умові length, а також пропущені product_category_name, замовлення без review, статуси shipped/unavailable без фінальної дати доставки, потенційні дублікати review_id на один order_id.

customer_unique_id обрано, бо один реальний покупець може мати кілька customer_id — інакше фічі й сегментація розпорошуються між окремими клієнтами.

seller_score потребував грануляності seller_id × order_id, бо order_items дублює замовлення по кількості товарів — без дедуплікації середній рейтинг і лічильник review були б спотворені.

In [25]:
engine.dispose()
db.cleanup()
print("Сервер зупинено.")

Сервер зупинено.
